# Unit 6 — Evaluating the U-Net

We measure two things:

1. **Quality** on the clean test set — per-sample Dice and IoU, plus a qualitative grid
   of the best and worst predictions.
2. **Adversarial robustness** — we add Gaussian noise to the input at several strengths
   and watch IoU decay.

In [ ]:
import os, glob
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
torch.manual_seed(0); np.random.seed(0)

## 1. Load the trained model and test data

In [ ]:
ROOT = "datasets/bbbc039_seg"
CKPT = "checkpoints/unet_efficientnet-b0.pt"

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

model = smp.Unet(encoder_name="efficientnet-b0", encoder_weights=None,
                 in_channels=3, classes=1).to(device)
state = torch.load(CKPT, map_location=device, weights_only=False)
model.load_state_dict(state["model"])
model.eval()
print(f"Loaded checkpoint from epoch {state['epoch']}  (val Dice={state['val_dice']:.4f})")

img_paths  = sorted(glob.glob(os.path.join(ROOT, "test", "images", "*.png")))
mask_paths = sorted(glob.glob(os.path.join(ROOT, "test", "masks",  "*.png")))
print(f"Test pairs: {len(img_paths)}")

## 2. Helpers

`run_model` does the whole pipeline in numpy: image in [0,1] RGB → tensor → logits → sigmoid → binary mask.

In [ ]:
def load_image_uint8(p):
    return np.array(Image.open(p).convert("RGB"))      # (H,W,3) uint8

def load_mask_uint8(p):
    return (np.array(Image.open(p).convert("L")) > 127).astype(np.uint8)   # (H,W) {0,1}

@torch.no_grad()
def run_model(images_uint8):
    """images_uint8: (N,H,W,3) uint8. Returns sigmoid probs (N,H,W) float32 in [0,1]."""
    x = images_uint8.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    x = torch.from_numpy(x.transpose(0, 3, 1, 2)).to(device)
    logits = model(x)
    return torch.sigmoid(logits).squeeze(1).cpu().numpy()

def dice_iou(pred, gt):
    """pred, gt: (H,W) binary uint8."""
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    psum  = pred.sum() + gt.sum()
    dice = (2 * inter) / psum if psum > 0 else 1.0
    iou  = inter / union      if union > 0 else 1.0
    return float(dice), float(iou)

## 3. Clean test-set evaluation

Loop over every test pair, predict, score.

In [ ]:
results = []
for ip, mp in zip(img_paths, mask_paths):
    img = load_image_uint8(ip)
    gt  = load_mask_uint8(mp)
    prob = run_model(img[None])[0]
    pred = (prob > 0.5).astype(np.uint8)
    d, i = dice_iou(pred, gt)
    results.append({"base": os.path.basename(ip), "dice": d, "iou": i,
                    "img": img, "gt": gt, "pred": pred, "prob": prob})

dices = np.array([r["dice"] for r in results])
ious  = np.array([r["iou"]  for r in results])
print(f"Dice  = {dices.mean():.4f} ± {dices.std():.4f}   "
      f"min={dices.min():.4f}  max={dices.max():.4f}")
print(f"IoU   = {ious.mean():.4f} ± {ious.std():.4f}   "
      f"min={ious.min():.4f}  max={ious.max():.4f}")

### Distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(dices, bins=20, color="C0", edgecolor="k")
ax[0].axvline(dices.mean(), color="r", ls="--", label=f"mean={dices.mean():.3f}")
ax[0].set_title("Dice"); ax[0].set_xlabel("Dice"); ax[0].legend()
ax[1].hist(ious, bins=20, color="C2", edgecolor="k")
ax[1].axvline(ious.mean(), color="r", ls="--", label=f"mean={ious.mean():.3f}")
ax[1].set_title("IoU"); ax[1].set_xlabel("IoU"); ax[1].legend()
plt.tight_layout(); plt.show()

### Best and worst samples

In [ ]:
def show_row(r, ax):
    ax[0].imshow(r["img"]);                ax[0].set_title("Input")
    ax[1].imshow(r["gt"],  cmap="gray");   ax[1].set_title("Ground truth")
    ax[2].imshow(r["pred"], cmap="gray");  ax[2].set_title("Prediction")
    ax[3].imshow(r["prob"], cmap="magma", vmin=0, vmax=1); ax[3].set_title("Prob")
    for a in ax: a.axis("off")

order = sorted(results, key=lambda r: r["iou"])
worst = order[:2]
best  = order[-2:][::-1]

fig, axes = plt.subplots(4, 4, figsize=(13, 12))
for i, r in enumerate(best):
    show_row(r, axes[i])
    axes[i, 0].set_ylabel(f"BEST  IoU={r['iou']:.3f}")
for i, r in enumerate(worst):
    show_row(r, axes[i + 2])
    axes[i + 2, 0].set_ylabel(f"WORST IoU={r['iou']:.3f}")
plt.tight_layout(); plt.show()

## 4. Robustness under Gaussian noise

We add zero-mean Gaussian noise to the input in [0,1] space at increasing **σ**:
0.00 → 0.30. At each level we re-evaluate IoU over all test samples.

In [ ]:
SIGMAS = [0.0, 0.001, 0.002, 0.005, 0.010]
rng = np.random.default_rng(0)

per_sigma_iou  = []
per_sigma_dice = []

# Pre-load images once
imgs = np.stack([r["img"] for r in results]).astype(np.float32) / 255.0    # (N,H,W,3) in [0,1]
gts  = np.stack([r["gt"]  for r in results])                                # (N,H,W) {0,1}

BATCH = 4
for sigma in SIGMAS:
    noisy = imgs + rng.normal(0, sigma, imgs.shape).astype(np.float32)
    noisy = np.clip(noisy, 0, 1)
    noisy_u8 = (noisy * 255).astype(np.uint8)
    probs = []
    for i in range(0, len(noisy_u8), BATCH):
        probs.append(run_model(noisy_u8[i:i + BATCH]))
    probs = np.concatenate(probs, axis=0)
    preds = (probs > 0.5).astype(np.uint8)
    ds, is_ = [], []
    for p, g in zip(preds, gts):
        d, i = dice_iou(p, g)
        ds.append(d); is_.append(i)
    per_sigma_dice.append(np.array(ds))
    per_sigma_iou.append(np.array(is_))
    print(f"σ={sigma:.3f}  mean IoU={np.mean(is_):.4f}  mean Dice={np.mean(ds):.4f}")

### IoU and Dice vs noise level

In [ ]:
means_iou  = [a.mean() for a in per_sigma_iou]
means_dice = [a.mean() for a in per_sigma_dice]
stds_iou   = [a.std()  for a in per_sigma_iou]
stds_dice  = [a.std()  for a in per_sigma_dice]

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.errorbar(SIGMAS, means_iou,  yerr=stds_iou,  marker="o", capsize=3, label="IoU")
ax.errorbar(SIGMAS, means_dice, yerr=stds_dice, marker="s", capsize=3, label="Dice")
ax.set_xlabel("Gaussian noise σ (input space, [0,1])")
ax.set_ylabel("score")
ax.set_title("Segmentation quality vs input noise")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

### one sample at each noise level

Pick the median-quality sample and show how its prediction collapses as σ grows.

In [ ]:
# Pick the median sample by clean IoU
median_idx = int(np.argsort([r["iou"] for r in results])[len(results) // 2])
img_clean = imgs[median_idx]               # (H,W,3) in [0,1]
gt        = gts[median_idx]

cols = len(SIGMAS)
fig, axes = plt.subplots(2, cols, figsize=(2.2 * cols, 4.6))
for j, sigma in enumerate(SIGMAS):
    noisy = np.clip(img_clean + rng.normal(0, sigma, img_clean.shape).astype(np.float32), 0, 1)
    noisy_u8 = (noisy * 255).astype(np.uint8)
    prob = run_model(noisy_u8[None])[0]
    pred = (prob > 0.5).astype(np.uint8)
    iou  = dice_iou(pred, gt)[1]
    axes[0, j].imshow(noisy);              axes[0, j].set_title(f"σ={sigma:.3f}")
    axes[1, j].imshow(pred, cmap="gray");  axes[1, j].set_title(f"IoU={iou:.2f}")
    for a in (axes[0, j], axes[1, j]): a.axis("off")
axes[0, 0].set_ylabel("Noisy input", fontsize=10)
axes[1, 0].set_ylabel("Prediction",  fontsize=10)
plt.suptitle(f"Median test sample under noise — base {results[median_idx]['base']}",
             fontsize=12)
plt.tight_layout(); plt.show()

## 5. Discussion

* **Where does the model break first?**
* **Mitigations:** noise-augmented training??? could help

## Summary

Performance is necessary but not sufficient. The robustness evaluation is useful and important for any deployment
decision.